# Figuras para o Artigo — TCC MBA ENAP
## Monitoramento e Diagnóstico de Eventos Disruptivos nos Portos Brasileiros

Notebook unificado de figuras estáticas com qualidade de publicação.

- **Parte A** — 16 figuras oficiais do roteiro (PNG 300 DPI + PDF vetorial)
- **Parte B** — Figuras complementares do dashboard (PNG 300 DPI)

**Estilo:** Times New Roman 10pt, fundo branco, paleta sóbria funcional em P&B.

**Largura máxima:** 16 cm (~6.3 in) para coluna A4 ENAP.

In [ ]:
# ── Cell 1: Setup ────────────────────────────────────────────────────
import warnings; warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
import matplotlib.patches as mpatches
from matplotlib.path import Path as MPath
from matplotlib.patches import PathPatch
from pathlib import Path
import json

# Plotnine
from plotnine import *

# Sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

# ── Paths ──
ROOT = Path('c:/Users/fabio/OneDrive - Ministério da Gestão e da Inovação dos Serv. Pub/Fábio - Projetos CGEC/MBA/TCC MBA ENAP')
DATA_PROC = ROOT / 'artigo' / 'data' / 'processed'
DATA_OUT  = ROOT / 'artigo' / 'data' / 'output'
DASH_DATA = ROOT / 'artigo' / 'dashboard' / 'data'
FIG_DIR   = ROOT / 'artigo' / 'outputs' / 'figuras'
FIG_DIR.mkdir(parents=True, exist_ok=True)

# ── Estilo global (roteiro) ──
FONT = 'Times New Roman'
FONT_FB = 'DejaVu Serif'
try:
    from matplotlib.font_manager import findfont, FontProperties
    fp = findfont(FontProperties(family=FONT))
    if 'times' not in fp.lower():
        FONT = FONT_FB
except Exception:
    FONT = FONT_FB

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': [FONT, FONT_FB, 'serif'],
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': False,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
})

# ── Paleta (roteiro) ──
PAL = {
    'global': '#C0392B',
    'nacional': '#E67E22',
    'isolado': '#3498DB',
    'observado': '#2C3E50',
    'predito': '#7F8C8D',
    'anomalia': '#E74C3C',
    'grid': '#BDC3C7',
    'light_gray': '#BDC3C7',
    'bg_gray': '#F0F0F0',
}

# Plotnine academic theme
theme_academic = (
    theme_bw(base_family=FONT, base_size=10)
    + theme(
        plot_background=element_rect(fill='white', color='none'),
        panel_background=element_rect(fill='white'),
        panel_grid_major=element_line(color='#E0E0E0', size=0.3, alpha=0.3),
        panel_grid_minor=element_blank(),
        axis_ticks=element_line(color='#333333', size=0.4),
        plot_title=element_text(size=11, weight='bold', ha='left'),
        legend_background=element_rect(fill='white', color='#CCCCCC'),
        strip_background=element_rect(fill='#F5F5F5', color='#CCCCCC'),
    )
)

# ── Helpers ──
MAX_W = 6.3  # max width in inches (~16cm)

def save_fig(fig, name, w=6.3, h=4):
    """Save plotnine figure as PNG + PDF."""
    for ext in ['png', 'pdf']:
        path = FIG_DIR / f'{name}.{ext}'
        fig.save(path, width=w, height=h, dpi=300, verbose=False)
    print(f'Salvo: {name}.png + .pdf')

def save_mpl(fig, name):
    """Save matplotlib figure as PNG + PDF."""
    for ext in ['png', 'pdf']:
        path = FIG_DIR / f'{name}.{ext}'
        fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white',
                    pad_inches=0.05)
    print(f'Salvo: {name}.png + .pdf')

print('\u2713 Setup completo \u2014', FONT)

In [ ]:
# ── Cell 2: Carregar dados ────────────────────────────────────────────

# Helper: fix Mojibake (UTF-8 encoded as latin-1)
def fix_mojibake(s):
    """Attempt to fix double-encoded strings: latin1→utf8."""
    if not isinstance(s, str):
        return s
    try:
        return s.encode('latin-1').decode('utf-8')
    except (UnicodeDecodeError, UnicodeEncodeError):
        return s

def fix_df_strings(df):
    """Fix all string columns in a DataFrame."""
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].map(fix_mojibake)
    return df

# Parquet
features = pd.read_parquet(DATA_PROC / 'features_semanal.parquet')
features['date'] = pd.to_datetime(features['date'])

residuos = pd.read_parquet(DATA_OUT / 'residuos.parquet')
residuos['date'] = pd.to_datetime(residuos['date'])

anomalias = pd.read_parquet(DATA_OUT / 'anomalias_classificadas.parquet')
anomalias['date'] = pd.to_datetime(anomalias['date'])

fingerprints = pd.read_parquet(DATA_OUT / 'fingerprints.parquet')
fingerprints['date'] = pd.to_datetime(fingerprints['date'])

indice_global = pd.read_parquet(DATA_PROC / 'indice_global.parquet')
indice_global['date'] = pd.to_datetime(indice_global['date'])

# CSV (with Mojibake fix)
vuln_matriz = fix_df_strings(pd.read_csv(DATA_OUT / 'comex_v2_matriz_vulnerabilidade.csv'))
hhi_porto = fix_df_strings(pd.read_csv(DATA_OUT / 'comex_v2_hhi_porto.csv'))
hhi_cadeia = fix_df_strings(pd.read_csv(DATA_OUT / 'comex_v2_hhi_cadeia.csv'))
hhi_uf = fix_df_strings(pd.read_csv(DATA_OUT / 'comex_v2_hhi_uf_rich.csv'))
uf_cuci_porto = fix_df_strings(pd.read_csv(DATA_OUT / 'comex_v2_uf_cuci_porto.csv'))
matriz_uf_porto = fix_df_strings(pd.read_csv(DATA_OUT / 'comex_matriz_uf_porto.csv'))
perfil_cuci_porto = fix_df_strings(pd.read_csv(DATA_OUT / 'comex_v2_perfil_cuci_porto.csv'))

# Geo
import geopandas as gpd
br_states = gpd.read_file(DASH_DATA / 'br_states.topojson')
portos_meta = pd.read_json(DASH_DATA / 'portos_meta.json')

# Constantes
DIMS = ['atracacoes', 'tonelagem_exp', 't1_mediano', 'tatracado_mediano']
DIM_LABELS = {
    'atracacoes': 'Atracações/semana',
    'tonelagem_exp': 'Tonelagem exportação (t)',
    't1_mediano': 'Espera p/ atracar (h)',
    'tatracado_mediano': 'Tempo atracado (h)',
}

# Ordem geográfica norte → sul
PORTO_ORDER = [
    'Vila do Conde', 'São Luís', 'Manaus', 'Pecém', 'Salvador', 'Suape',
    'Barra do Riacho', 'Vitória', 'São João da Barra', 'Rio de Janeiro',
    'Itaguaí', 'Santos', 'Paranaguá', 'São Francisco do Sul', 'Itajaí',
    'Imbituba', 'Rio Grande', 'Porto Alegre',
]

# Remover última semana incompleta (2026-W05: apenas 6 dias, dados ANTAQ não consolidados)
_CUTOFF = pd.Timestamp('2026-01-19')
features     = features[features['date']     <= _CUTOFF]
anomalias    = anomalias[anomalias['date']   <= _CUTOFF]
residuos     = residuos[residuos['date']     <= _CUTOFF]
fingerprints = fingerprints[fingerprints['date'] <= _CUTOFF] if 'date' in fingerprints.columns else fingerprints

print(f'Dados: {len(features)} features, {len(residuos)} resíduos, '
      f'{len(anomalias)} anomalias, {len(fingerprints)} fingerprints')
print('Encoding fix: OK —', vuln_matriz['NO_CUCI_GRUPO'].iloc[0][:30])

---
## Figura 1 — Séries temporais do Porto de Santos
**§3.1** — Facet vertical 4 painéis, eixo x compartilhado.
Eventos: Greve Caminhoneiros, COVID-19, Enchentes RS.

In [ ]:
# ── Figura 1: Séries temporais Santos ─────────────────────────────────
santos = features[features['porto'] == 'Santos'].copy()
santos = santos[santos['date'] >= '2015-06-01'].sort_values('date')

EVENTS = [
    ('2018-05-21', '2018-06-10', 'Greve\nCaminhoneiros', '#C0392B'),
    ('2020-03-01', '2020-06-30', 'COVID-19', '#8E44AD'),
    ('2024-04-25', '2024-07-15', 'Enchentes RS', '#27AE60'),
]

dim_list = [
    ('atracacoes', 'Atraca\u00e7\u00f5es / semana', None),
    ('tonelagem_exp', 'Tonelagem exp.\n(milh\u00f5es t)', 1e6),
    ('t1_mediano', 'Espera p/ atracar (h)', None),
    ('tatracado_mediano', 'Tempo atracado (h)', None),
]

fig, axes = plt.subplots(4, 1, figsize=(6.3, 7), sharex=True,
                          gridspec_kw={'hspace': 0.08})

for i, (dim, ylabel, divisor) in enumerate(dim_list):
    ax = axes[i]
    y = santos[dim].values.copy()
    if divisor:
        y = y / divisor
    ax.plot(santos['date'], y, linewidth=0.8, color=PAL['observado'], alpha=0.9)

    # Event spans
    for start, end, label, color in EVENTS:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                   alpha=0.12, color=color, zorder=0)
        if i == 0:  # labels only on top panel
            mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2
            ax.text(mid, ax.get_ylim()[1] * 0.92, label, ha='center',
                    fontsize=7, color=color, fontweight='bold', va='top')

    ax.set_ylabel(ylabel, fontsize=9)
    ax.grid(axis='y', alpha=0.2, linewidth=0.4)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

axes[-1].xaxis.set_major_locator(mdates.YearLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[-1].set_xlabel('')

fig.suptitle('Porto de Santos \u2014 S\u00e9ries temporais semanais (2015\u20132025)',
             fontsize=11, fontweight='bold', x=0.02, ha='left', y=0.98)
fig.align_ylabels(axes)

save_mpl(fig, 'fig01_series_santos')
plt.show()

---
## Figura 2 — Modelo preditivo: observado vs. predito + resíduos
**§3.1** — Painel vertical 2×1 (Santos, atracações).

In [ ]:
# ── Figura 2: Observado vs Predito + Resíduos ────────────────────────
res_s = residuos[(residuos['porto'] == 'Santos') & (residuos['dim'] == 'atracacoes')].copy()
anom_s = anomalias[(anomalias['porto'] == 'Santos') & (anomalias['dim'] == 'atracacoes')].copy()
anom_dates = set(anom_s['date'])
res_s['is_anom'] = res_s['date'].isin(anom_dates)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.3, 4.3), height_ratios=[1.5, 1],
                                sharex=True, gridspec_kw={'hspace': 0.06})

# Panel 1: Observed vs Predicted
ax1.plot(res_s['date'], res_s['y_true'], linewidth=0.7, color=PAL['observado'],
         alpha=0.85, label='Observado')
ax1.plot(res_s['date'], res_s['y_pred'], linewidth=0.7, color=PAL['predito'],
         linestyle='--', alpha=0.8, label='Predito (XGBoost)')
ax1.set_ylabel('Atraca\u00e7\u00f5es / semana')
ax1.legend(fontsize=8, frameon=True, edgecolor='#CCCCCC', loc='upper right')
ax1.set_title('Santos \u2014 Atraca\u00e7\u00f5es: observado vs. predito e res\u00edduos',
              fontsize=11, fontweight='bold', loc='left')
ax1.grid(axis='y', alpha=0.2, linewidth=0.4)

# Panel 2: Residuals
normal = res_s[~res_s['is_anom']]
anom_r = res_s[res_s['is_anom']]
ax2.vlines(normal['date'], 0, normal['innovation'], colors=PAL['light_gray'],
           linewidth=0.3, alpha=0.6)
ax2.vlines(anom_r['date'], 0, anom_r['innovation'], colors=PAL['anomalia'],
           linewidth=1.0, alpha=0.8)
ax2.axhline(0, color='#999999', linewidth=0.5)
ax2.set_ylabel('Inova\u00e7\u00e3o AR(1)')
ax2.set_xlabel('')
ax2.grid(axis='y', alpha=0.2, linewidth=0.4)

ax2.legend(
    handles=[
        Line2D([0],[0], color=PAL['light_gray'], lw=1.5, label='Normal'),
        Line2D([0],[0], color=PAL['anomalia'], lw=2, label='Anomalia (ensemble \u2265 2/3)'),
    ],
    fontsize=7, frameon=True, edgecolor='#CCCCCC', loc='lower right'
)

for ax in [ax1, ax2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.text(0.99, -0.02, 'XGBoost + AR(1) \u00b7 Walk-forward \u00b7 Ensemble \u2265 2/3',
         ha='right', fontsize=7, color='#888888')

save_mpl(fig, 'fig02_modelo_residuos')
plt.show()

---
## Figura 3 — Scatter de classificação dual (Score A × Score B)
**§3.2** — n=1.609 pontos, 3 classes, legendas de cor e tamanho.

In [ ]:
# ── Figura 3: Scatter Score A × Score B ───────────────────────────────
anom = anomalias.copy()
anom['score_b'] = anom['score_b'].fillna(0)

CLS_LABELS = {'global': 'Global', 'nacional': 'Nacional', 'isolado': 'Isolado'}
anom['cls'] = anom['classificacao'].map(CLS_LABELS)

# n_dims per porto-date
dim_counts = anom.groupby(['porto', 'date']).size().reset_index(name='n_dims')
anom = anom.merge(dim_counts, on=['porto', 'date'], how='left')

# Jitter
np.random.seed(42)
anom['sa_j'] = anom['score_a'] + np.random.uniform(-0.3, 0.3, len(anom))
anom['sb_j'] = anom['score_b'] + np.random.uniform(-0.05, 0.05, len(anom))

fig = (
    ggplot(anom, aes(x='sa_j', y='sb_j', color='cls', size='n_dims'))
    + geom_point(alpha=0.30, stroke=0)
    + geom_vline(xintercept=3, linetype='dashed', color='#999999', size=0.4)
    + geom_hline(yintercept=0.8, linetype='dashed', color='#999999', size=0.4)
    + annotate('text', x=1.5, y=-0.12, label='ISOLADO',
               color=PAL['isolado'], size=9, fontweight='bold', alpha=0.35)
    + annotate('text', x=10, y=-0.12, label='NACIONAL',
               color=PAL['nacional'], size=9, fontweight='bold', alpha=0.35)
    + annotate('text', x=9, y=2.5, label='GLOBAL',
               color=PAL['global'], size=9, fontweight='bold', alpha=0.35)
    + annotate('text', x=3.2, y=-0.25, label='t_a = 3', size=7, color='#666666', ha='left')
    + annotate('text', x=0.3, y=0.88, label='t_b = 0,8', size=7, color='#666666', ha='left')
    + scale_color_manual(
        values={'Global': PAL['global'], 'Isolado': PAL['isolado'], 'Nacional': PAL['nacional']},
        name='Classifica\u00e7\u00e3o'
    )
    + scale_size_continuous(range=(0.8, 4), breaks=[1, 2, 4], name='Dimens\u00f5es\nafetadas')
    + labs(
        x='Score A (co-ocorr\u00eancia entre portos)',
        y='Score B (\u00edndice de choque global)',
        title='Classifica\u00e7\u00e3o das anomalias: Score A \u00d7 Score B',
    )
    + theme_academic
    + theme(legend_position='right', legend_direction='vertical',
            figure_size=(5.9, 4.3))
)
save_fig(fig, 'fig03_scatter_scores', w=5.9, h=4.3)
fig.draw(); plt.show()

---
## Figura 4 — Fingerprints operacionais comparativos
**§3.2** — 4 radares (Greve, COVID, Enchentes RS, Ataques Houthi).

In [ ]:
# ── Figura 4: Fingerprints Radar (2×2) ────────────────────────────────
EVENT_DEFS = {
    'Greve Caminhoneiros': ('2018-05-21', '2018-06-10', '#C0392B'),
    'COVID-19':            ('2020-03-01', '2020-06-30', '#8E44AD'),
    'Enchentes RS':        ('2024-04-25', '2024-06-30', '#27AE60'),
    'Ataques Houthi':      ('2023-11-15', '2024-03-31', '#2980B9'),
}

dims_radar = ['atracacoes', 'tonelagem_exp', 't1_mediano', 'tatracado_mediano']
labels_radar = ['Atraca\u00e7\u00f5es', 'Tonelagem\nExporta\u00e7\u00e3o', 'Espera p/\nAtracar', 'Tempo\nAtracado']

def make_radar(ax, values, title, color):
    N = len(values)
    angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
    vals = list(values) + [values[0]]
    angles_closed = angles + [angles[0]]
    ax.plot(angles_closed, vals, 'o-', lw=1.5, ms=4, color=color)
    ax.fill(angles_closed, vals, alpha=0.20, color=color)
    ax.set_ylim(0, 1.15)
    ax.set_xticks(angles)
    ax.set_xticklabels(labels_radar, fontsize=7)
    ax.set_yticklabels([])
    for g in [0.25, 0.50, 0.75, 1.00]:
        ax.plot(np.linspace(0, 2*np.pi, 100), [g]*100, '-',
                color='#DDDDDD', linewidth=0.3)
    ax.set_title(title, fontsize=9, fontweight='bold', pad=12)

fig, axes = plt.subplots(2, 2, figsize=(5.5, 5.5), subplot_kw=dict(polar=True))
axes = axes.flatten()

for idx, (ev_name, (ev_start, ev_end, ev_color)) in enumerate(EVENT_DEFS.items()):
    fp = fingerprints[
        (fingerprints['date'] >= pd.Timestamp(ev_start)) &
        (fingerprints['date'] <= pd.Timestamp(ev_end))
    ]
    n_anom = len(fp)
    means = fp[dims_radar].mean().values
    max_v = means.max() if means.max() > 0 else 1
    normed = means / max_v
    title = f'{ev_name} (n={n_anom})'
    make_radar(axes[idx], normed, title, ev_color)

fig.suptitle('Fingerprints operacionais por evento', fontsize=11,
             fontweight='bold', y=1.02)
fig.tight_layout()
save_mpl(fig, 'fig04_fingerprints_radar')
plt.show()

---
## Figura 5 — Heatmap de anomalias por porto × ano
**§3.2** — 18 portos × 10 anos (2016–2025), paleta YlOrRd.

In [ ]:
# ── Figura 5: Heatmap anomalias porto × ano ──────────────────────────
anom_h = anomalias.copy()
anom_h['year'] = anom_h['date'].dt.year
anom_h = anom_h[anom_h['year'] >= 2016]

# Count unique anomaly events per porto-year
events_hm = anom_h.drop_duplicates(['porto', 'date']).groupby(
    ['porto', 'year']).size().reset_index(name='n')

# Pivot
heat = events_hm.pivot(index='porto', columns='year', values='n').fillna(0).astype(int)
heat = heat.reindex([p for p in PORTO_ORDER if p in heat.index])
years = sorted(heat.columns)
heat = heat[years]

fig, ax = plt.subplots(figsize=(6.3, 5.5))

cmap = plt.cm.YlOrRd
im = ax.imshow(heat.values, cmap=cmap, aspect='auto', vmin=0,
               vmax=heat.values.max())

# Annotations
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        val = heat.values[i, j]
        if val > 0:
            text_color = 'white' if val > heat.values.max() * 0.65 else '#333333'
            ax.text(j, i, str(val), ha='center', va='center',
                    fontsize=7.5, color=text_color, fontweight='bold')

ax.set_xticks(range(len(years)))
ax.set_xticklabels(years, fontsize=9)
ax.set_yticks(range(len(heat.index)))
ax.set_yticklabels(heat.index, fontsize=8.5)
ax.set_title('Anomalias detectadas por porto e ano (2016\u20132025)',
             fontsize=11, fontweight='bold', loc='left')

cbar = fig.colorbar(im, ax=ax, shrink=0.7, pad=0.02)
cbar.set_label('Anomalias detectadas', fontsize=9)

for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(0.5)

fig.tight_layout()
save_mpl(fig, 'fig05_heatmap_anomalias')
plt.show()

---
## Figura 6 — Vulnerabilidade cruzada HHI (porto × cadeia)
**§3.3** — Scatter com bolhas (FOB), quadrantes de concentração.

In [ ]:
# ── Figura 6: Vulnerabilidade cruzada HHI ─────────────────────────────
from adjustText import adjust_text

cs = vuln_matriz.copy()
cs['fob_bi'] = cs['VL_FOB'] / 1e9

def quadrant(row):
    if row['hhi_pauta'] >= 0.20 and row['hhi_portuario'] >= 0.20:
        return 'Dupla concentra\u00e7\u00e3o'
    elif row['hhi_pauta'] < 0.20 and row['hhi_portuario'] < 0.20:
        return 'Diversificado'
    else:
        return 'Concentra\u00e7\u00e3o parcial'

cs['quadrante'] = cs.apply(quadrant, axis=1)

top = cs.sort_values('VL_FOB', ascending=False).head(10).copy()
top['label'] = top['porto'].str[:10] + '\n' + top['NO_CUCI_GRUPO'].str[:18]

fig_mpl, ax = plt.subplots(figsize=(5.9, 4.3))

# Shade high-risk quadrant
ax.fill_between([0.20, cs['hhi_pauta'].max() + 0.05], 0.20,
                cs['hhi_portuario'].max() + 0.05,
                alpha=0.06, color=PAL['anomalia'], zorder=0)

colors_q = {'Dupla concentra\u00e7\u00e3o': PAL['global'],
            'Diversificado': '#27AE60',
            'Concentra\u00e7\u00e3o parcial': PAL['isolado']}
for q, grp in cs.groupby('quadrante'):
    ax.scatter(grp['hhi_pauta'], grp['hhi_portuario'],
              s=grp['fob_bi'] * 1.5,
              c=colors_q[q], alpha=0.5, edgecolors='none', label=q)

ax.axvline(0.20, ls='--', color='#999999', lw=0.5)
ax.axhline(0.20, ls='--', color='#999999', lw=0.5)

texts = []
for _, r in top.iterrows():
    texts.append(ax.text(r['hhi_pauta'], r['hhi_portuario'],
                         r['label'], fontsize=5.5, color='#333333'))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='#999999', lw=0.3),
            force_text=0.4, force_points=0.3)

ax.set_xlabel('HHI do porto (concentra\u00e7\u00e3o da pauta exportadora)')
ax.set_ylabel('HHI da cadeia (depend\u00eancia portu\u00e1ria)')
ax.set_title('Vulnerabilidade cruzada: porto \u00d7 cadeia produtiva',
             fontsize=11, fontweight='bold', loc='left')

# Combined legend
handles_q = [mpatches.Patch(color=colors_q[q], alpha=0.6, label=q) for q in colors_q]
for sz_val, sz_label in [(5, '5'), (50, '50'), (200, '200')]:
    handles_q.append(plt.scatter([], [], s=sz_val * 1.5, c='gray', alpha=0.5,
                                  label=f'{sz_label} bi US$'))
ax.legend(handles=handles_q, fontsize=6.5, frameon=True, edgecolor='#CCCCCC',
          loc='upper left', ncol=2, columnspacing=0.5, handletextpad=0.3)

save_mpl(fig_mpl, 'fig06_vulnerabilidade_cruzada')
plt.show()

---
## Figura 7 — Mapa de fluxos UF → Porto (MT, PA, MG)
**§3.3** — Arcos curvos no mapa do Brasil, 3 UFs destacadas.

In [ ]:
# ── Figura 7: Mapa de fluxos UF → Porto ──────────────────────────────
from shapely.geometry import Point

# UF centroids
uf_centroids = br_states.copy()
uf_centroids['centroid'] = uf_centroids.geometry.centroid
uf_map = {}
for _, row in uf_centroids.iterrows():
    uf_code = row['id'][-2:]
    uf_map[uf_code] = (row['centroid'].x, row['centroid'].y)

# Porto coordinates
porto_coords = {row['porto']: (row['lon'], row['lat'])
                for _, row in portos_meta.iterrows()}

UFS = ['MT', 'PA', 'MG']
uf_colors = {'MT': '#E67E22', 'PA': '#27AE60', 'MG': '#2980B9'}
uf_labels_full = {'MT': 'MT (HHI=37%)', 'PA': 'PA (HHI=85%)', 'MG': 'MG (HHI=21%)'}

# Map UF -> state id
uf_to_stateid = {}
for _, row in br_states.iterrows():
    for uf in UFS:
        if row['id'].endswith(uf):
            uf_to_stateid[uf] = row['id']

fig, ax = plt.subplots(figsize=(5.5, 6.3))

# Background
br_states.plot(ax=ax, color='#F0F0F0', edgecolor='#CCCCCC', linewidth=0.4)

# Highlight UFs
for uf in UFS:
    sid = uf_to_stateid.get(uf)
    if sid:
        br_states[br_states['id'] == sid].plot(
            ax=ax, color=mcolors.to_rgba(uf_colors[uf], 0.15),
            edgecolor=uf_colors[uf], linewidth=1.2)

# Draw arcs
for uf in UFS:
    uf_row = matriz_uf_porto[matriz_uf_porto['SG_UF_NCM'] == uf]
    if len(uf_row) == 0:
        continue
    uf_row = uf_row.iloc[0]
    origin = uf_map.get(uf)
    if not origin:
        continue

    port_cols = [c for c in matriz_uf_porto.columns if c != 'SG_UF_NCM']
    flows = [(p, uf_row[p]) for p in port_cols if p in porto_coords and uf_row[p] > 0.02]
    flows.sort(key=lambda x: x[1], reverse=True)
    flows = flows[:5]

    for porto_name, pct in flows:
        dest = porto_coords.get(porto_name)
        if not dest:
            continue
        lw = max(0.5, pct * 8)
        alpha = min(0.8, 0.3 + pct)
        arrow = FancyArrowPatch(
            origin, dest,
            connectionstyle='arc3,rad=0.2',
            arrowstyle='->,head_length=3,head_width=2',
            color=uf_colors[uf], lw=lw, alpha=alpha, zorder=5
        )
        ax.add_patch(arrow)

# Porto points
for porto_name, (lon, lat) in porto_coords.items():
    ax.plot(lon, lat, 'o', color='#2C3E50', ms=3, zorder=6)
    ax.text(lon + 0.4, lat, porto_name, fontsize=5, color='#333333',
            va='center', zorder=7)

# UF labels
for uf in UFS:
    c = uf_map.get(uf)
    if c:
        ax.text(c[0], c[1], uf_labels_full[uf], fontsize=7,
                fontweight='bold', color=uf_colors[uf], ha='center',
                va='center', zorder=8,
                bbox=dict(boxstyle='round,pad=0.2', fc='white', ec=uf_colors[uf],
                          alpha=0.8, lw=0.5))

ax.set_xlim(-75, -33)
ax.set_ylim(-35, 6)
ax.set_aspect('equal')
ax.axis('off')
ax.set_title('Fluxos de exporta\u00e7\u00e3o: UF \u2192 Porto (MT, PA, MG)',
             fontsize=11, fontweight='bold', loc='left')

for uf in UFS:
    ax.plot([], [], '-', color=uf_colors[uf], lw=2, label=uf_labels_full[uf])
ax.legend(fontsize=7, frameon=True, edgecolor='#CCCCCC', loc='lower left')

save_mpl(fig, 'fig07_mapa_fluxos')
plt.show()

---
## Figura 8 — Diagrama Sankey UF → Porto
**§3.3** — Implementação matplotlib com retângulos e curvas Bézier.

In [ ]:
# ── Figura 8: Sankey UF → Porto (matplotlib) ─────────────────────────
UFS_SANKEY = ['MT', 'PA', 'MG']
uf_colors_sk = {'MT': '#E67E22', 'PA': '#27AE60', 'MG': '#2980B9'}

uf_fobs = {}
for uf in UFS_SANKEY:
    row = hhi_uf[hhi_uf['SG_UF_NCM'] == uf]
    if len(row) > 0:
        uf_fobs[uf] = row.iloc[0]['fob_total']

flows_data = []
for uf in UFS_SANKEY:
    uf_row = matriz_uf_porto[matriz_uf_porto['SG_UF_NCM'] == uf]
    if len(uf_row) == 0:
        continue
    uf_row = uf_row.iloc[0]
    total_fob = uf_fobs.get(uf, 1)
    port_cols = [c for c in matriz_uf_porto.columns if c != 'SG_UF_NCM']
    for p in port_cols:
        pct = uf_row[p]
        if pct >= 0.03:
            flows_data.append((uf, p, pct * total_fob, pct))

# Port totals
porto_total = {}
for uf, p, fob, pct in flows_data:
    porto_total[p] = porto_total.get(p, 0) + fob
portos_sorted = sorted(porto_total.keys(), key=lambda x: porto_total[x], reverse=True)

fig, ax = plt.subplots(figsize=(6.3, 4))

left_x = 0.05
right_x = 0.75
node_w = 0.08

# Left nodes (UFs)
left_total = sum(uf_fobs.get(uf, 0) for uf in UFS_SANKEY)
left_positions = {}
y_cursor = 0.05
for uf in UFS_SANKEY:
    h = (uf_fobs.get(uf, 0) / left_total) * 0.85
    left_positions[uf] = (y_cursor, y_cursor + h)
    rect = FancyBboxPatch((left_x, y_cursor), node_w, h,
                           boxstyle='round,pad=0.005',
                           facecolor=uf_colors_sk[uf], edgecolor='white', alpha=0.85)
    ax.add_patch(rect)
    mid_y = y_cursor + h / 2
    fob_bi = uf_fobs.get(uf, 0) / 1e9
    ax.text(left_x - 0.01, mid_y, f'{uf}\nUS$ {fob_bi:.0f} bi',
            ha='right', va='center', fontsize=8, fontweight='bold',
            color=uf_colors_sk[uf])
    y_cursor += h + 0.03

# Right nodes
right_total = sum(porto_total[p] for p in portos_sorted)
right_positions = {}
y_cursor = 0.02
for p in portos_sorted:
    h = (porto_total[p] / right_total) * 0.90
    h = max(h, 0.015)
    right_positions[p] = (y_cursor, y_cursor + h)
    rect = FancyBboxPatch((right_x, y_cursor), node_w, h,
                           boxstyle='round,pad=0.005',
                           facecolor='#2C3E50', edgecolor='white', alpha=0.7)
    ax.add_patch(rect)
    mid_y = y_cursor + h / 2
    fob_bi = porto_total[p] / 1e9
    ax.text(right_x + node_w + 0.01, mid_y, f'{p} ({fob_bi:.0f} bi)',
            ha='left', va='center', fontsize=6.5)
    y_cursor += h + 0.01

# Flow curves
def draw_flow(ax, x0, y0_s, y0_e, x1, y1_s, y1_e, color, alpha=0.3):
    verts = [
        (x0, y0_s), (0.4, y0_s), (0.4, y1_s), (x1, y1_s),
        (x1, y1_e), (0.4, y1_e), (0.4, y0_e), (x0, y0_e),
        (x0, y0_s),
    ]
    codes = [MPath.MOVETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
             MPath.LINETO, MPath.CURVE4, MPath.CURVE4, MPath.CURVE4,
             MPath.CLOSEPOLY]
    path = MPath(verts, codes)
    patch = PathPatch(path, facecolor=color, edgecolor='none', alpha=alpha)
    ax.add_patch(patch)

left_cursors = {uf: left_positions[uf][0] for uf in UFS_SANKEY}
right_cursors = {p: right_positions[p][0] for p in portos_sorted}

for uf, porto, fob, pct in sorted(flows_data, key=lambda x: x[2], reverse=True):
    if porto not in right_positions:
        continue
    uf_h = left_positions[uf][1] - left_positions[uf][0]
    flow_h_left = pct * uf_h
    y0_s = left_cursors[uf]
    y0_e = y0_s + flow_h_left
    left_cursors[uf] = y0_e

    porto_h = right_positions[porto][1] - right_positions[porto][0]
    flow_h_right = (fob / porto_total[porto]) * porto_h
    y1_s = right_cursors[porto]
    y1_e = y1_s + flow_h_right
    right_cursors[porto] = y1_e

    draw_flow(ax, left_x + node_w, y0_s, y0_e,
              right_x, y1_s, y1_e,
              color=uf_colors_sk[uf], alpha=0.25)

ax.set_xlim(-0.15, 1.0)
ax.set_ylim(-0.02, 1.05)
ax.axis('off')
ax.set_title('Fluxos de exporta\u00e7\u00e3o: UF \u2192 Porto principal',
             fontsize=11, fontweight='bold', loc='left')

save_mpl(fig, 'fig08_sankey_uf_porto')
plt.show()

---
## Figura 9 — Tabela MAE: naïve vs. XGBoost
**§3.1** — Tabela formatada como figura.

In [ ]:
# ── Figura 9: Tabela MAE ──────────────────────────────────────────────
table_data = [
    ['Atracações/sem', '3,96', '0,73', '↓ 82%', '429'],
    ['Tonelagem exp.', '172.760', '41.329', '↓ 76%', '477'],
    ['Espera p/ atracar (h)', '37,29', '10,08', '↓ 73%', '316'],
    ['Tempo no berço (h)', '15,13', '3,28', '↓ 78%', '387'],
]
col_labels = ['Dimens\u00e3o', 'MAE Na\u00efve', 'MAE XGBoost', 'Redu\u00e7\u00e3o', 'Anomalias']

fig, ax = plt.subplots(figsize=(6.3, 1.8))
ax.axis('off')

table = ax.table(
    cellText=table_data,
    colLabels=col_labels,
    loc='center',
    cellLoc='center',
)

table.auto_set_font_size(False)
table.set_fontsize(9)
table.scale(1.0, 1.6)

for j in range(len(col_labels)):
    cell = table[0, j]
    cell.set_facecolor('#2C3E50')
    cell.set_text_props(color='white', fontweight='bold', fontsize=9)

for i in range(len(table_data)):
    for j in range(len(col_labels)):
        cell = table[i + 1, j]
        cell.set_facecolor('#F8F8F8' if i % 2 == 0 else 'white')
        cell.set_edgecolor('#CCCCCC')

ax.set_title('Desempenho preditivo: Na\u00efve vs. XGBoost (walk-forward)',
             fontsize=11, fontweight='bold', loc='left', pad=15)

fig.text(0.5, -0.05,
         'Nota: anomalias contadas por dimens\u00e3o \u2014 uma mesma semana-porto pode ter anomalia em mais de uma dimens\u00e3o.',
         ha='center', fontsize=7, color='#888888', style='italic')

save_mpl(fig, 'fig09_tabela_mae')
plt.show()

---
## Figura 10 — Tabela dos 10 eventos de calibração
**§2.3** — Tabela formatada como figura.

In [ ]:
# ── Figura 10: Tabela de eventos de calibração ───────────────────────
events_cal = [
    ['1', 'Greve Caminhoneiros', 'Mai\u2013Jun 2018', 'Nacional', '4'],
    ['2', 'COVID-19', 'Mar\u2013Jun 2020', 'Global', '18'],
    ['3', 'COVID recupera\u00e7\u00e3o', 'Set/20\u2013Jun/21', 'Global', '18'],
    ['4', 'Bloqueio Suez', 'Mar\u2013Abr 2021', 'Global', '18'],
    ['5', 'Guerra Ucr\u00e2nia', 'Fev\u2013Dez 2022', 'Global', '18'],
    ['6', 'Lockdowns China', 'Mar\u2013Jun 2022', 'Global', '18'],
    ['7', 'Ataques Houthi', 'Nov/23\u2013Jun/24', 'Global', '18'],
    ['8', 'Enchentes RS', 'Abr\u2013Jun 2024', 'Isolado', '2'],
    ['9', 'Greve Receita Fed.', 'Dez/24\u2013Jan/25', 'Nacional', '3'],
    ['10', 'Tarifa\u00e7\u00e3o EUA', 'Fev\u2013Jun 2025', 'Global', '18'],
]
cols_ev = ['#', 'Evento', 'Per\u00edodo', 'Classifica\u00e7\u00e3o', 'Portos']

fig, ax = plt.subplots(figsize=(6.3, 3.5))
ax.axis('off')

table = ax.table(
    cellText=events_cal,
    colLabels=cols_ev,
    loc='center',
    cellLoc='center',
    colWidths=[0.05, 0.30, 0.22, 0.22, 0.10],
)

table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1.0, 1.45)

for j in range(len(cols_ev)):
    cell = table[0, j]
    cell.set_facecolor('#2C3E50')
    cell.set_text_props(color='white', fontweight='bold', fontsize=8)

cls_colors = {'Global': PAL['global'], 'Nacional': PAL['nacional'], 'Isolado': PAL['isolado']}
for i, row in enumerate(events_cal):
    cls = row[3]
    cell = table[i + 1, 3]
    cell.set_text_props(color=cls_colors.get(cls, '#333333'), fontweight='bold')
    for j in range(len(cols_ev)):
        cell_ij = table[i + 1, j]
        cell_ij.set_facecolor('#F8F8F8' if i % 2 == 0 else 'white')
        cell_ij.set_edgecolor('#CCCCCC')

ax.set_title('Eventos de calibra\u00e7\u00e3o do sistema de classifica\u00e7\u00e3o',
             fontsize=11, fontweight='bold', loc='left', pad=15)

save_mpl(fig, 'fig10_tabela_eventos')
plt.show()

---
## Figura 11 — Clusters de portos (PCA biplot)
**§3.2** — PC1 × PC2 com labels de porto e cores por cluster k=4.

In [ ]:
# ── Figura 11: Clusters de portos (PCA biplot) ───────────────────────
from adjustText import adjust_text

# Compute per-port features (same as NB8)
ser = features[features['date'] >= '2015-06-01'].copy()
port_feats = []
for porto in ser['porto'].unique():
    ps = ser[ser['porto'] == porto]
    row = {'porto': porto}
    for dim in DIMS:
        vals = ps[dim].dropna()
        row[f'{dim}_mean'] = vals.mean()
        row[f'{dim}_cv'] = vals.std() / vals.mean() if vals.mean() > 0 else 0
        if len(vals) > 2:
            row[f'{dim}_acf1'] = vals.autocorr(lag=1)
        else:
            row[f'{dim}_acf1'] = 0
    port_feats.append(row)

pf = pd.DataFrame(port_feats)
feat_cols = [c for c in pf.columns if c != 'porto']

# Standardize + PCA
scaler = StandardScaler()
X = scaler.fit_transform(pf[feat_cols])
pca = PCA(n_components=5)
pc = pca.fit_transform(X)
pf['PC1'] = pc[:, 0]
pf['PC2'] = pc[:, 1]

# K-Means k=4
km = KMeans(n_clusters=4, random_state=42, n_init=10)
pf['cluster'] = km.fit_predict(pc[:, :5])

# Identify cluster roles
means_by_cluster = pf.groupby('cluster')['atracacoes_mean'].mean()
cv_by_cluster = pf.groupby('cluster')['atracacoes_cv'].mean()
hub_cl = means_by_cluster.idxmax()
micro_cl = pf.loc[pf['atracacoes_mean'].idxmin(), 'cluster']

cluster_names = {}
for cl in range(4):
    n = len(pf[pf['cluster'] == cl])
    if cl == hub_cl:
        cluster_names[cl] = f'Hub exportador (n={n})'
    elif cl == micro_cl and n <= 2:
        cluster_names[cl] = f'Microporto (n={n})'
    elif cv_by_cluster[cl] > 0.4:
        cluster_names[cl] = f'Nicho vol\u00e1til (n={n})'
    else:
        cluster_names[cl] = f'Generalista (n={n})'

pf['cluster_label'] = pf['cluster'].map(cluster_names)

var1 = pca.explained_variance_ratio_[0] * 100
var2 = pca.explained_variance_ratio_[1] * 100
var5 = pca.explained_variance_ratio_[:5].sum() * 100

cluster_colors = ['#C0392B', '#3498DB', '#27AE60', '#8E44AD']

fig, ax = plt.subplots(figsize=(5.5, 4.7))

for cl in sorted(pf['cluster'].unique()):
    subset = pf[pf['cluster'] == cl]
    ax.scatter(subset['PC1'], subset['PC2'], s=60,
              c=cluster_colors[cl % len(cluster_colors)],
              alpha=0.7, edgecolors='white', linewidths=0.5,
              label=cluster_names[cl], zorder=5)

texts = []
for _, row in pf.iterrows():
    texts.append(ax.text(row['PC1'], row['PC2'], row['porto'],
                         fontsize=6, color='#333333'))
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='#999999', lw=0.3),
            force_text=0.5)

# Loading arrows
loadings = pca.components_[:2].T
scale = 3
for i, feat in enumerate(feat_cols):
    mag = np.sqrt(loadings[i, 0]**2 + loadings[i, 1]**2)
    if mag > 0.35:
        ax.annotate('', xy=(loadings[i, 0]*scale, loadings[i, 1]*scale),
                    xytext=(0, 0),
                    arrowprops=dict(arrowstyle='->', color='#888888', lw=0.8))
        ax.text(loadings[i, 0]*scale*1.1, loadings[i, 1]*scale*1.1,
                feat.replace('_', '\n'), fontsize=5.5, color='#666666',
                ha='center')

ax.axhline(0, color='#DDDDDD', lw=0.5)
ax.axvline(0, color='#DDDDDD', lw=0.5)
ax.set_xlabel(f'PC1 ({var1:.1f}%)', fontsize=10)
ax.set_ylabel(f'PC2 ({var2:.1f}%)', fontsize=10)
ax.set_title(f'Clusters de portos \u2014 PCA biplot (5 PCs, {var5:.1f}% var.)',
             fontsize=11, fontweight='bold', loc='left')
ax.legend(fontsize=7, frameon=True, edgecolor='#CCCCCC', loc='best')

save_mpl(fig, 'fig11_clusters_pca')
plt.show()

---
## Figura 12 — Perfis de disrupção (clusters de semanas anômalas)
**§3.2** — Grouped bar chart com z-scores médios por dimensão/cluster.

In [ ]:
# ── Figura 12: Perfis de disrupção ────────────────────────────────────
# Use anomalias table to identify anomalous (porto, date) pairs
anom_keys = anomalias[['porto', 'date']].drop_duplicates()
res_anom = residuos.merge(anom_keys, on=['porto', 'date'], how='inner')

res_wide = res_anom.pivot_table(index=['porto', 'date'], columns='dim',
                                 values='innovation', aggfunc='first').dropna()
res_wide.columns.name = None

scaler_w = StandardScaler()
X_w = scaler_w.fit_transform(res_wide[DIMS])

km_w = KMeans(n_clusters=4, random_state=42, n_init=10)
res_wide['week_cluster'] = km_w.fit_predict(X_w)

profiles = []
for cl in range(4):
    mask = res_wide['week_cluster'] == cl
    subset_x = X_w[mask.values]
    row = {'cluster': cl, 'n': mask.sum()}
    for j, dim in enumerate(DIMS):
        row[dim] = subset_x[:, j].mean()
    profiles.append(row)

prof_df = pd.DataFrame(profiles)

# Assign descriptive labels based on dominant dimension
for i, row in prof_df.iterrows():
    if row['tatracado_mediano'] > 0.5:
        prof_df.loc[i, 'label'] = f"Gargalo extremo (n={int(row['n'])})"
    elif row['tatracado_mediano'] < -0.5:
        prof_df.loc[i, 'label'] = f"Eficiência extrema (n={int(row['n'])})"
    elif row['atracacoes'] > 0.3:
        prof_df.loc[i, 'label'] = f"Pico de volume (n={int(row['n'])})"
    else:
        prof_df.loc[i, 'label'] = f"Queda de volume (n={int(row['n'])})"

prof_long = prof_df.melt(id_vars=['cluster', 'n', 'label'],
                          value_vars=DIMS, var_name='dim', value_name='z_mean')
prof_long['dim_label'] = prof_long['dim'].map(DIM_LABELS)

cluster_pal = ['#C0392B', '#E67E22', '#3498DB', '#27AE60']

fig = (
    ggplot(prof_long, aes(x='dim_label', y='z_mean', fill='label'))
    + geom_col(position='dodge', width=0.7, alpha=0.85)
    + geom_hline(yintercept=0, color='#999999', size=0.4)
    + scale_fill_manual(values=dict(zip(prof_df['label'], cluster_pal[:len(prof_df)])),
                        name='Perfil de disrupção')
    + labs(
        x='',
        y='Z-score médio dos resíduos',
        title='Perfis de disrupção das semanas anômalas (K-Means k=4)',
    )
    + theme_academic
    + theme(
        axis_text_x=element_text(rotation=25, ha='right', size=8),
        figure_size=(6.3, 3.8),
        legend_position='right',
        legend_direction='vertical',
    )
)
save_fig(fig, 'fig12_perfis_disrupcao', w=6.3, h=3.8)
fig.draw(); plt.show()

---
## Figura 13 — Diagnóstico de autocorrelação (ACF antes/depois do AR(1))
**§3.1** — Scatter |ACF(1)| antes vs. após, com diagonal de referência.

In [ ]:
# ── Figura 13: Diagnóstico ACF antes/depois AR(1) ────────────────────
acf_data = []
for (porto, dim), grp in residuos.groupby(['porto', 'dim']):
    if len(grp) < 20:
        continue
    acf1_resid = grp['residual'].autocorr(lag=1)
    acf1_innov = grp['innovation'].autocorr(lag=1)
    acf_data.append({
        'porto': porto, 'dim': dim,
        'acf1_resid': abs(acf1_resid) if pd.notna(acf1_resid) else 0,
        'acf1_innov': abs(acf1_innov) if pd.notna(acf1_innov) else 0,
    })

acf_df = pd.DataFrame(acf_data)

med_before = acf_df['acf1_resid'].median()
med_after = acf_df['acf1_innov'].median()
n_improved = (acf_df['acf1_innov'] < acf_df['acf1_resid']).sum()
pct_below_015 = (acf_df['acf1_innov'] < 0.15).mean() * 100

fig, ax = plt.subplots(figsize=(4.7, 4.7))

ax.plot([0, 0.7], [0, 0.7], '--', color='#CCCCCC', lw=0.8, zorder=1)
ax.axvline(0.15, ls=':', color='#DDDDDD', lw=0.5)
ax.axhline(0.15, ls=':', color='#DDDDDD', lw=0.5)

dim_colors_acf = {'atracacoes': '#C0392B', 'tonelagem_exp': '#E67E22',
                  't1_mediano': '#3498DB', 'tatracado_mediano': '#27AE60'}
for dim in DIMS:
    sub = acf_df[acf_df['dim'] == dim]
    ax.scatter(sub['acf1_resid'], sub['acf1_innov'], s=30,
              c=dim_colors_acf[dim], alpha=0.7, edgecolors='white',
              linewidths=0.3, label=DIM_LABELS[dim], zorder=3)

ax.text(0.5, 0.02, f'Mediana antes: {med_before:.2f}\n'
        f'Mediana depois: {med_after:.2f}\n'
        f'{n_improved}/{len(acf_df)} pares melhoraram\n'
        f'{pct_below_015:.0f}% com |ACF(1)| < 0,15',
        fontsize=7, color='#666666', transform=ax.transAxes,
        va='bottom', ha='center',
        bbox=dict(boxstyle='round,pad=0.3', fc='white', ec='#CCCCCC', alpha=0.9))

ax.set_xlabel('|ACF(1)| dos res\u00edduos (antes AR(1))')
ax.set_ylabel('|ACF(1)| das inova\u00e7\u00f5es (ap\u00f3s AR(1))')
ax.set_title('Diagn\u00f3stico de autocorrela\u00e7\u00e3o: efeito do filtro AR(1)',
             fontsize=11, fontweight='bold', loc='left')
ax.legend(fontsize=7, frameon=True, edgecolor='#CCCCCC', loc='upper right')
ax.set_xlim(-0.02, 0.65)
ax.set_ylim(-0.02, 0.65)
ax.set_aspect('equal')

save_mpl(fig, 'fig13_diagnostico_acf')
plt.show()

---
## Figura 14 — Índice global PortWatch e z-score
**§2.3** — Painel 2×1: portcalls/dia (topo), z-score (baixo).

In [ ]:
# ── Figura 14: Índice global PortWatch ────────────────────────────────
gi = indice_global.copy()

EVENTS_GI = [
    ('2020-03-01', '2020-06-30', 'COVID-19', '#C0392B'),
    ('2021-03-20', '2021-04-15', 'Suez', '#E67E22'),
    ('2022-02-24', '2022-06-30', 'Ucr\u00e2nia', '#8E44AD'),
    ('2022-03-01', '2022-06-30', 'Lockdowns\nChina', '#27AE60'),
    ('2023-11-15', '2024-03-31', 'Ataques\nHouthi', '#2980B9'),
]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(6.3, 4), sharex=True,
                                gridspec_kw={'hspace': 0.08, 'height_ratios': [1, 1]})

ax1.plot(gi['date'], gi['gi_portcalls'], lw=0.7, color=PAL['observado'], alpha=0.8)
ax1.set_ylabel('Portcalls/dia\n(top 50 n\u00e3o-BR)', fontsize=8)
ax1.set_title('\u00cdndice de choque global \u2014 PortWatch/FMI',
              fontsize=11, fontweight='bold', loc='left')
ax1.grid(axis='y', alpha=0.2, lw=0.4)

ax2.plot(gi['date'], gi['gi_z'], lw=0.7, color=PAL['observado'], alpha=0.8)
ax2.fill_between(gi['date'], 0, gi['gi_z'],
                 where=gi['gi_z'].abs() > 0.8,
                 alpha=0.12, color=PAL['anomalia'], interpolate=True)
ax2.axhline(0, color='#999999', lw=0.4)
ax2.axhline(0.8, color=PAL['anomalia'], lw=0.4, ls='--', alpha=0.5)
ax2.axhline(-0.8, color=PAL['anomalia'], lw=0.4, ls='--', alpha=0.5)
ax2.text(gi['date'].min(), 0.9, 't_b = 0,8', fontsize=7, color=PAL['anomalia'], alpha=0.7)
ax2.set_ylabel('Z-score (12 sem.)', fontsize=8)
ax2.set_xlabel('')
ax2.grid(axis='y', alpha=0.2, lw=0.4)

for start, end, label, color in EVENTS_GI:
    for ax in [ax1, ax2]:
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                   alpha=0.08, color=color, zorder=0)
    mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2
    ax1.text(mid, ax1.get_ylim()[1] * 0.95, label, ha='center',
             fontsize=6, color=color, fontweight='bold', va='top')

for ax in [ax1, ax2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

fig.text(0.99, -0.03, 'Fonte: PortWatch/FMI \u00b7 Top 50 portos n\u00e3o-BR \u00b7 M\u00e9dia semanal de portcalls',
         ha='right', fontsize=6.5, color='#888888')

save_mpl(fig, 'fig14_indice_global')
plt.show()

---
## Figura 15 — Top cadeias expostas por porto (stacked bar horizontal)
**§3.3** — Top 5 portos × top 5 cadeias CUCI.

In [ ]:
# ── Figura 15: Top cadeias por porto (stacked bar) ───────────────────
top_portos = ['Santos', 'Itagua\u00ed', 'Paranagu\u00e1', 'Rio Grande', 'Vila do Conde']

rows = []
for porto in top_portos:
    sub = perfil_cuci_porto[perfil_cuci_porto['porto'] == porto].copy()
    sub = sub.sort_values('pct', ascending=False).head(5)
    for _, r in sub.iterrows():
        rows.append({
            'porto': porto,
            'cadeia': str(r['NO_CUCI_GRUPO'])[:25],
            'pct': r['pct'] * 100,
            'fob_bi': r['fob'] / 1e9,
        })
    other_pct = max(0, 100 - sub['pct'].sum() * 100)
    rows.append({'porto': porto, 'cadeia': 'Outros', 'pct': other_pct, 'fob_bi': 0})

df_stack = pd.DataFrame(rows)

sector_colors = [
    '#C0392B', '#E67E22', '#3498DB', '#27AE60', '#8E44AD', '#BDC3C7',
]

fig, ax = plt.subplots(figsize=(6.3, 3.5))

porto_y = {p: i for i, p in enumerate(top_portos)}
for porto in top_portos:
    sub = df_stack[df_stack['porto'] == porto]
    left = 0
    for idx, (_, row) in enumerate(sub.iterrows()):
        color = sector_colors[min(idx, len(sector_colors) - 1)]
        ax.barh(porto_y[porto], row['pct'], left=left, height=0.6,
                color=color, alpha=0.8, edgecolor='white', linewidth=0.3)
        if row['pct'] > 6:
            ax.text(left + row['pct'] / 2, porto_y[porto],
                    f"{row['cadeia'][:15]}\n{row['pct']:.0f}%",
                    ha='center', va='center', fontsize=5, color='white',
                    fontweight='bold')
        left += row['pct']

ax.set_yticks(list(porto_y.values()))
ax.set_yticklabels(top_portos, fontsize=9)
ax.set_xlabel('% da pauta de exporta\u00e7\u00e3o')
ax.set_title('Composi\u00e7\u00e3o da pauta exportadora \u2014 Top 5 portos',
             fontsize=11, fontweight='bold', loc='left')
ax.set_xlim(0, 105)
ax.invert_yaxis()
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

save_mpl(fig, 'fig15_cadeias_porto')
plt.show()

---
## Figura 16 — Painel de Alerta (semana exemplo)
**§3.4** — Tabela com badge de classificação (ALERTA/VIGILÂNCIA/NORMAL).

In [ ]:
# ── Figura 16: Painel de Alerta ───────────────────────────────────────
alerta_data = [
    ('Santos', -3.53, 'ALERTA'),
    ('Itagua\u00ed', -2.25, 'ALERTA'),
    ('S\u00e3o Lu\u00eds', -1.86, 'VIGIL\u00c2NCIA'),
    ('Rio de Janeiro', -1.82, 'VIGIL\u00c2NCIA'),
    ('Imbituba', -1.74, 'VIGIL\u00c2NCIA'),
    ('Paranagu\u00e1', -1.71, 'VIGIL\u00c2NCIA'),
    ('S\u00e3o Jo\u00e3o da Barra', -1.62, 'VIGIL\u00c2NCIA'),
    ('Salvador', -1.41, 'NORMAL'),
    ('Pec\u00e9m', -1.22, 'NORMAL'),
    ('Itaja\u00ed', -1.15, 'NORMAL'),
    ('Vila do Conde', +0.85, 'NORMAL'),
]

badge_colors = {
    'ALERTA': '#C0392B',
    'VIGIL\u00c2NCIA': '#E67E22',
    'NORMAL': '#27AE60',
}

fig, ax = plt.subplots(figsize=(5, 4.2))
ax.axis('off')

table_data = []
for porto, zscore, badge in alerta_data:
    table_data.append([porto, f'{zscore:+.2f}\u03c3', badge])

cols_al = ['Porto', 'Z-score', 'Status']
table = ax.table(
    cellText=table_data,
    colLabels=cols_al,
    loc='center',
    cellLoc='center',
    colWidths=[0.40, 0.25, 0.30],
)

table.auto_set_font_size(False)
table.set_fontsize(8.5)
table.scale(1.0, 1.4)

for j in range(len(cols_al)):
    cell = table[0, j]
    cell.set_facecolor('#2C3E50')
    cell.set_text_props(color='white', fontweight='bold', fontsize=9)

for i, (porto, zscore, badge) in enumerate(alerta_data):
    cell = table[i + 1, 2]
    cell.set_text_props(color='white', fontweight='bold')
    cell.set_facecolor(badge_colors[badge])

    cell_z = table[i + 1, 1]
    if zscore < -2:
        cell_z.set_text_props(color=PAL['global'], fontweight='bold')
    elif zscore < -1.5:
        cell_z.set_text_props(color=PAL['nacional'])

    for j in range(len(cols_al)):
        if j != 2:
            cell_ij = table[i + 1, j]
            cell_ij.set_facecolor('#F8F8F8' if i % 2 == 0 else 'white')
            cell_ij.set_edgecolor('#CCCCCC')
        table[i + 1, j].set_edgecolor('#CCCCCC')

ax.set_title('Painel de Alerta \u2014 Semana de 09/mar/2026',
             fontsize=11, fontweight='bold', loc='left', pad=15)

fig.text(0.5, -0.03, 'Nota: z-scores negativos indicam queda de portcalls. '
         'Poss\u00edvel efeito da tarifa\u00e7\u00e3o EUA (evento #10).',
         ha='center', fontsize=7, color='#888888', style='italic')

save_mpl(fig, 'fig16_painel_alerta')
plt.show()

---
## Parte A — Resumo das figuras oficiais (roteiro)

| Fig. | Arquivo | Seção | Descrição |
|------|---------|-------|----------|
| 1 | `fig01_series_santos` | §3.1 | Séries temporais Santos (4 dimensões) |
| 2 | `fig02_modelo_residuos` | §3.1 | Observado vs. predito + resíduos (Santos) |
| 3 | `fig03_scatter_scores` | §3.2 | Scatter Score A × Score B (n=1.588) |
| 4 | `fig04_fingerprints_radar` | §3.2 | Fingerprints operacionais (4 eventos) |
| 5 | `fig05_heatmap_anomalias` | §3.2 | Heatmap porto × ano (2016–2025) |
| 6 | `fig06_vulnerabilidade_cruzada` | §3.3 | Scatter HHI porto × cadeia |
| 7 | `fig07_mapa_fluxos` | §3.3 | Mapa UF → Porto (MT, PA, MG) |
| 8 | `fig08_sankey_uf_porto` | §3.3 | Diagrama Sankey UF → Porto |
| 9 | `fig09_tabela_mae` | §3.1 | Tabela MAE Naïve vs. XGBoost |
| 10 | `fig10_tabela_eventos` | §2.3 | 10 eventos de calibração |
| 11 | `fig11_clusters_pca` | §3.2 | PCA biplot clusters de portos |
| 12 | `fig12_perfis_disrupcao` | §3.2 | Perfis de disrupção K-Means |
| 13 | `fig13_diagnostico_acf` | §3.1 | Diagnóstico autocorrelação AR(1) |
| 14 | `fig14_indice_global` | §2.3 | Índice PortWatch portcalls + z-score |
| 15 | `fig15_cadeias_porto` | §3.3 | Composição pauta exportadora top 5 |
| 16 | `fig16_painel_alerta` | §3.4 | Painel de alerta (semana exemplo) |

Figuras oficiais em `v3/outputs/figuras_v2/` — PNG 300 DPI + PDF vetorial.

---
# Parte B — Figuras Complementares

Figuras adicionais geradas a partir dos dados do dashboard (JSONs).
Complementam as figuras oficiais do roteiro com visualizações de apoio.

**Nota:** Estas figuras requerem os dados JSON do dashboard (`dashboard/data/`).
O setup de dados abaixo carrega esses JSONs.

In [ ]:
# ── Dados do dashboard (para figuras complementares) ──────────────
# ── Carregar todos os dados ────────────────────────────────────────────────
series       = pd.read_json(DASH_DATA / 'series.json')
anomalias    = pd.read_json(DASH_DATA / 'anomalias.json')
residuos     = pd.read_json(DASH_DATA / 'residuos.json')
fingerprints = pd.read_json(DASH_DATA / 'fingerprints.json')
gi           = pd.read_json(DASH_DATA / 'indice_global.json')
vuln         = pd.read_json(DASH_DATA / 'vulnerabilidade.json')
portos_meta  = pd.read_json(DASH_DATA / 'portos_meta.json')
comex_hhi    = pd.read_json(DASH_DATA / 'comex_hhi.json')
export_via   = pd.read_json(DASH_DATA / 'export_via_ano.json')
cuci_scatter = pd.read_json(DASH_DATA / 'cuci_scatter.json')
cuci_rank_fob = pd.read_json(DASH_DATA / 'cuci_ranking_cadeias_fob.json')
fob_exp_uf   = pd.read_json(DASH_DATA / 'fob_exposto_uf.json')
fob_exp_cls  = pd.read_json(DASH_DATA / 'fob_exposto_porto_cls.json')
comex_flows  = pd.read_json(DASH_DATA / 'comex_flows.json')

# Converter datas
for df in [series, anomalias, residuos, fingerprints]:
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])

print(f'Dados carregados: {len(series)} séries, {len(anomalias)} anomalias, {len(portos_meta)} portos')

---
## Figura 1 — Via marítima nas exportações brasileiras
Participação do transporte marítimo no total exportado (FOB), 2014–2025.

In [ ]:
# ── Figura 1: Via marítima nas exportações ─────────────────────────────────
eva = export_via.sort_values('ano').copy()
eva['fob_total_bi'] = eva['fob_total'] / 1e9
eva['fob_maritima_bi'] = eva['fob_maritima'] / 1e9
eva['fob_outras_bi'] = eva['fob_total_bi'] - eva['fob_maritima_bi']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(7, 5), height_ratios=[1, 1.2],
                                sharex=True, gridspec_kw={'hspace': 0.08})

# ── Painel superior: % marítima ──
ax1.plot(eva['ano'], eva['pct_maritima'], '-o', color=PAL['blue'],
         markersize=5, linewidth=1.8, zorder=3)
ax1.fill_between(eva['ano'], eva['pct_maritima'].min() - 1, eva['pct_maritima'],
                 alpha=0.08, color=PAL['blue'])
for _, row in eva.iterrows():
    ax1.annotate(f"{row['pct_maritima']:.1f}%",
                 (row['ano'], row['pct_maritima']),
                 textcoords='offset points', xytext=(0, 10),
                 ha='center', fontsize=7.5, color=PAL['dark'])
ax1.set_ylabel('Participação marítima (%)')
ax1.set_ylim(78, 92)
ax1.yaxis.set_major_formatter(mtick.PercentFormatter())
ax1.set_title('Via marítima nas exportações brasileiras (FOB, 2014–2025)',
              fontsize=11, fontweight='bold', loc='left')

# ── Painel inferior: FOB (US$ bi) ──
ax2.bar(eva['ano'], eva['fob_maritima_bi'], width=0.55,
        color=PAL['blue'], alpha=0.7, label='Via marítima')
ax2.bar(eva['ano'], eva['fob_outras_bi'], width=0.55,
        bottom=eva['fob_maritima_bi'],
        color=PAL['light_gray'], alpha=0.6, label='Demais vias')
ax2.set_ylabel('FOB exportado (US$ bilhões)')
ax2.set_xlabel('Ano')
ax2.legend(loc='upper left', frameon=True, framealpha=0.9, edgecolor='#ccc')
ax2.xaxis.set_major_locator(mtick.MaxNLocator(integer=True))

for ax in [ax1, ax2]:
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linewidth=0.5)

fig.align_ylabels()
fig.text(0.99, -0.02, 'Fonte: ComexStat/MDIC', ha='right', fontsize=7, color='#888')
save_mpl(fig, 'fig01_via_maritima')
plt.show()

---
## Figura 2 — Mapa dos 18 portos monitorados
Localização geográfica e volume médio de atracações semanais.

In [ ]:
# ── Figura 2: Mapa dos portos ─────────────────────────────────────────────
import geopandas as gpd
from adjustText import adjust_text

# Load Brazil states from local topojson
br_states = gpd.read_file(DASH_DATA / 'br_states.topojson')

# South America neighbors for context (from natural earth bundled in geopandas)
try:
    world = gpd.read_file(gpd.datasets.get_path('naturalearth_lowres'))
    sa = world[world['continent'] == 'South America']
except:
    sa = None

pm = portos_meta.copy()
pm_gdf = gpd.GeoDataFrame(pm, geometry=gpd.points_from_xy(pm.lon, pm.lat), crs='EPSG:4326')

fig, ax = plt.subplots(1, 1, figsize=(5.5, 7))

# Background: South America
if sa is not None:
    sa.plot(ax=ax, color='#f0f0f0', edgecolor='#cccccc', linewidth=0.3)

# Brazil UF boundaries
br_states.plot(ax=ax, color='#e8e8e8', edgecolor='#999999', linewidth=0.5)

# Port dots scaled by atracações
sizes = pm['atracacoes_media'] / pm['atracacoes_media'].max() * 200 + 15
ax.scatter(pm['lon'], pm['lat'], s=sizes, c=PAL['blue'],
           alpha=0.75, edgecolor='white', linewidth=0.6, zorder=5)

# Labels with adjustText to avoid overlap
texts = []
for _, row in pm.iterrows():
    t = ax.text(row['lon'], row['lat'], row['porto'],
                fontsize=7, ha='left', color=PAL['dark'])
    texts.append(t)
adjust_text(texts, ax=ax, arrowprops=dict(arrowstyle='-', color='#999', lw=0.4),
            expand=(1.5, 1.5))

# Size legend
for s_val, s_label in [(30, '30'), (100, '100'), (200, '200')]:
    s_size = s_val / pm['atracacoes_media'].max() * 200 + 15
    ax.scatter([], [], s=s_size, c=PAL['blue'], alpha=0.75,
               edgecolor='white', linewidth=0.6, label=f'{s_label} atracações/sem')
ax.legend(title='Volume médio', loc='lower left', fontsize=7, title_fontsize=8,
          frameon=True, framealpha=0.9, edgecolor='#ccc')

ax.set_xlim(-55, -30)
ax.set_ylim(-34, 2)
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.set_title('Portos monitorados — navegação de longo curso',
             fontsize=11, fontweight='bold', loc='left')
ax.text(0.99, -0.06, 'Fonte: ANTAQ (Anuário Estatístico Aquaviário)',
        transform=ax.transAxes, ha='right', fontsize=7, color='#888')

save_mpl(fig, 'fig02_mapa_portos')
plt.show()

---
## Figura 3 — Séries temporais operacionais
Quatro dimensões para os portos mais relevantes: Santos, Paranaguá e Rio Grande.

In [ ]:
# ── Figura 3: Séries temporais (Santos) ───────────────────────────────────
DIMS = ['atracacoes', 'tonelagem_exp', 't1_mediano', 'tatracado_mediano']
DIM_LABELS = {
    'atracacoes': 'Atracações/semana',
    'tonelagem_exp': 'Tonelagem de exportação (t)',
    't1_mediano': 'Espera p/ atracar (h)',
    'tatracado_mediano': 'Tempo no berço (h)',
}
DIM_COLORS = {
    'atracacoes': PAL['blue'],
    'tonelagem_exp': PAL['green'],
    't1_mediano': PAL['orange'],
    'tatracado_mediano': PAL['red'],
}

EVENTS = [
    ('2018-05-21', '2018-06-10', 'Greve Caminhoneiros'),
    ('2020-03-01', '2020-06-30', 'COVID-19'),
    ('2024-05-01', '2024-07-31', 'Enchentes RS'),
]
EVENT_COLORS = [PAL['red'], PAL['purple'], PAL['orange']]

def plot_series_porto(porto_name):
    data = series[series['porto'] == porto_name].copy()

    fig, axes = plt.subplots(4, 1, figsize=(7, 7), sharex=True,
                              gridspec_kw={'hspace': 0.12})

    for i, dim in enumerate(DIMS):
        ax = axes[i]
        ax.plot(data['date'], data[dim], linewidth=0.7,
                color=DIM_COLORS[dim], alpha=0.85)

        # Event shading
        for (start, end, label), ec in zip(EVENTS, EVENT_COLORS):
            ax.axvspan(pd.Timestamp(start), pd.Timestamp(end),
                       alpha=0.10, color=ec, zorder=0)
            if i == 0:
                mid = pd.Timestamp(start) + (pd.Timestamp(end) - pd.Timestamp(start)) / 2
                ax.text(mid, ax.get_ylim()[1] * 0.97, label,
                        fontsize=6.5, ha='center', color=ec, style='italic')

        ax.set_ylabel(DIM_LABELS[dim], fontsize=8)
        ax.tick_params(labelsize=8)
        ax.grid(axis='y', alpha=0.2, linewidth=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    axes[-1].set_xlabel('Data')
    axes[0].set_title(f'Séries temporais operacionais — {porto_name}',
                      fontsize=11, fontweight='bold', loc='left')

    # Event legend at bottom
    handles = [mpatches.Patch(facecolor=c, alpha=0.3, label=lbl)
               for (_, _, lbl), c in zip(EVENTS, EVENT_COLORS)]
    fig.legend(handles=handles, loc='lower center', ncol=3,
               fontsize=7.5, frameon=True, edgecolor='#ccc')

    fig.text(0.99, -0.01, 'Fonte: ANTAQ · Dados semanais 2014–2025',
             ha='right', fontsize=7, color='#888')

    return fig

# Santos
fig = plot_series_porto('Santos')
save_mpl(fig, 'fig03a_series_santos')
plt.show()

# Paranaguá
fig = plot_series_porto('Paranaguá')
save_mpl(fig, 'fig03b_series_paranagua')
plt.show()

# Rio Grande
fig = plot_series_porto('Rio Grande')
save_mpl(fig, 'fig03c_series_rio_grande')
plt.show()

---
## Figura 5b — Scatter Score A × Score B: Eventos destacados
Versões com destaque para eventos específicos.

In [ ]:
# ── Figura 5b: Scatter com eventos destacados ─────────────────────────────
EVENT_WINDOWS = {
    'Greve Caminhoneiros': ('2018-05-21', '2018-06-10'),
    'COVID-19': ('2020-03-01', '2020-06-30'),
    'Enchentes RS': ('2024-04-25', '2024-06-30'),
}

for ev_name, (ev_start, ev_end) in EVENT_WINDOWS.items():
    df = anom.copy()
    df['in_event'] = (df['date'] >= pd.Timestamp(ev_start)) & (df['date'] <= pd.Timestamp(ev_end))
    df['alpha_val'] = np.where(df['in_event'], 0.7, 0.06)

    fig_ev = (
        ggplot(df, aes(x='score_a_j', y='score_b_j', color='class_label'))
        + geom_point(aes(alpha='alpha_val'), size=2, stroke=0, show_legend=False)
        + geom_point(data=df[df['in_event']], size=2.5, alpha=0.8, stroke=0.3,
                     color=PAL['dark'])
        + geom_vline(xintercept=3, linetype='dashed', color='#999999', size=0.4)
        + geom_hline(yintercept=0.8, linetype='dashed', color='#999999', size=0.4)
        + scale_color_manual(values=[PAL['red'], PAL['blue'], PAL['orange']],
                             limits=['Global', 'Isolado', 'Nacional'])
        + scale_alpha_identity()
        + labs(
            x='Score A (co-ocorrência entre portos)',
            y='Score B (índice de choque global)',
            color='Classificação',
            title=f'Classificação das anomalias — destaque: {ev_name}',
        )
        + theme_academic
        + theme(legend_position='right', legend_direction='vertical', figure_size=(7, 5))
    )
    safe_name = ev_name.lower().replace(' ', '_').replace('-', '')
    save_fig(fig_ev, f'fig05b_scatter_{safe_name}', w=7, h=5)
    fig_ev.draw()
    plt.show()

---
## Figura 7b — Heatmap por classificação (global, nacional, isolado)
Versões separadas mostrando a distribuição por tipo de evento.

In [ ]:
# ── Figura 7b: Heatmaps por classificação ─────────────────────────────────
anom_h = anomalias.copy()
anom_h['year'] = anom_h['date'].dt.year
anom_h = anom_h[anom_h['year'] >= 2016]

cmaps_cls = {
    'global': LinearSegmentedColormap.from_list('red', ['#ffffff', '#fee0d2', '#fc9272', '#de2d26', '#a50f15']),
    'nacional': LinearSegmentedColormap.from_list('orange', ['#ffffff', '#fee6ce', '#fdae6b', '#e6550d', '#a63603']),
    'isolado': LinearSegmentedColormap.from_list('blue', ['#ffffff', '#deebf7', '#9ecae1', '#3182bd', '#08519c']),
}
cls_titles = {
    'global': 'Anomalias globais',
    'nacional': 'Anomalias nacionais',
    'isolado': 'Anomalias isoladas',
}

fig, axes = plt.subplots(1, 3, figsize=(12, 6), sharey=True,
                          gridspec_kw={'wspace': 0.05})

# Use same port order as main heatmap
heat_all = anom_h.groupby(['porto', 'year']).size().reset_index(name='count')
port_order = heat_all.groupby('porto')['count'].sum().sort_values(ascending=True).index.tolist()

for i, (cls, cmap) in enumerate(cmaps_cls.items()):
    ax = axes[i]
    subset = anom_h[anom_h['classificacao'] == cls]
    heat = subset.groupby(['porto', 'year']).size().reset_index(name='count')
    heat_pivot = heat.pivot(index='porto', columns='year', values='count').fillna(0)

    # Reindex to consistent ordering
    heat_pivot = heat_pivot.reindex(port_order).fillna(0)
    years = sorted(anom_h['year'].unique())
    for y in years:
        if y not in heat_pivot.columns:
            heat_pivot[y] = 0
    heat_pivot = heat_pivot[sorted(heat_pivot.columns)]

    vmax = max(heat_pivot.values.max(), 1)
    im = ax.imshow(heat_pivot.values, aspect='auto', cmap=cmap, vmin=0, vmax=vmax)

    if i == 0:
        ax.set_yticks(range(len(heat_pivot)))
        ax.set_yticklabels(heat_pivot.index, fontsize=7)
    ax.set_xticks(range(len(heat_pivot.columns)))
    ax.set_xticklabels(heat_pivot.columns.astype(int), fontsize=7, rotation=45)
    ax.set_title(cls_titles[cls], fontsize=10, fontweight='bold')

    for r in range(heat_pivot.shape[0]):
        for c in range(heat_pivot.shape[1]):
            val = int(heat_pivot.iloc[r, c])
            if val > 0:
                text_color = 'white' if val > vmax * 0.55 else '#333'
                ax.text(c, r, str(val), ha='center', va='center',
                        fontsize=6, color=text_color)

fig.suptitle('Anomalias por classificação, porto e ano (2016–2025)',
             fontsize=12, fontweight='bold')
fig.text(0.99, -0.01, 'Global: Score B ≥ 0,8 · Nacional: Score A ≥ 3 e B < 0,8 · Isolado: Score A < 3 e B < 0,8',
         ha='right', fontsize=7, color='#888')

save_mpl(fig, 'fig07b_heatmap_classificacao')
plt.show()

---
## Figura 8 — Concentração portuária por UF (HHI)
Grau de dependência de cada estado em relação a poucos portos.

In [ ]:
# ── Figura 8: HHI portuário por UF ────────────────────────────────────────
hhi = comex_hhi[~comex_hhi['SG_UF_NCM'].isin(['ND', 'CB'])].copy()
hhi = hhi.sort_values('hhi_portuario', ascending=False)
hhi['hhi_pct'] = hhi['hhi_portuario'] * 100

# Color categories
def hhi_category(v):
    if v >= 0.50: return 'Alta (HHI ≥ 50%)'
    elif v >= 0.35: return 'Moderada (35–50%)'
    else: return 'Diversificada (< 35%)'

hhi['categoria'] = hhi['hhi_portuario'].apply(hhi_category)
hhi['SG_UF_NCM'] = pd.Categorical(hhi['SG_UF_NCM'],
                                     categories=hhi['SG_UF_NCM'].tolist(),
                                     ordered=True)

fig = (
    ggplot(hhi, aes(x='SG_UF_NCM', y='hhi_pct', fill='categoria'))
    + geom_col(width=0.7, alpha=0.85)
    + geom_hline(yintercept=50, linetype='dashed', color='#999999', size=0.4)
    + geom_hline(yintercept=35, linetype='dotted', color='#999999', size=0.4)
    + geom_text(aes(label='hhi_pct'), format_string='{:.0f}%',
                va='bottom', size=7, nudge_y=1.5)
    + scale_fill_manual(
        values={'Alta (HHI ≥ 50%)': PAL['red'],
                'Moderada (35–50%)': PAL['orange'],
                'Diversificada (< 35%)': PAL['green']},
        name='Concentração'
    )
    + annotate('text', x=3, y=53, label='Alta concentração',
               color='#888888', size=7, fontstyle='italic')
    + annotate('text', x=3, y=37.5, label='Concentração moderada',
               color='#888888', size=7, fontstyle='italic')
    + labs(
        x='Unidade da Federação',
        y='HHI portuário (%)',
        title='Concentração portuária das exportações por UF',
    )
    + theme_academic
    + theme(
        axis_text_x=element_text(rotation=45, ha='right', size=8),
        figure_size=(8, 4.5),
        legend_position='top',
    )
)

save_fig(fig, 'fig08_hhi_uf', w=8, h=4.5)
fig.draw()
plt.show()

---
## Figura 9 — Vulnerabilidade dos portos
Ranking de vulnerabilidade composta dos 18 portos monitorados.

In [ ]:
# ── Figura 9: Ranking de vulnerabilidade dos portos ────────────────────────
v = vuln.copy()
v = v.sort_values('vulnerabilidade', ascending=True)
v['porto'] = pd.Categorical(v['porto'], categories=v['porto'].tolist(), ordered=True)

# Stacked bar: global + nacional + isolado
v_long = pd.melt(v, id_vars=['porto', 'vulnerabilidade'],
                 value_vars=['globais', 'nacionais', 'isoladas'],
                 var_name='tipo', value_name='n_anomalias')

tipo_labels = {'globais': 'Global', 'nacionais': 'Nacional', 'isoladas': 'Isolado'}
v_long['tipo_label'] = v_long['tipo'].map(tipo_labels)
v_long['tipo_label'] = pd.Categorical(v_long['tipo_label'],
                                        categories=['Global', 'Nacional', 'Isolado'],
                                        ordered=True)

fig = (
    ggplot(v_long, aes(x='porto', y='n_anomalias', fill='tipo_label'))
    + geom_col(width=0.7, alpha=0.85)
    + coord_flip()
    + scale_fill_manual(
        values={'Global': PAL['red'], 'Nacional': PAL['orange'], 'Isolado': PAL['blue']},
        name='Classificação',
    )
    + labs(
        x='',
        y='Número de anomalias detectadas',
        title='Anomalias por porto e classificação (2016–2025)',
    )
    + theme_academic
    + theme(
        figure_size=(7, 5.5),
        legend_position='bottom',
    )
)

save_fig(fig, 'fig09_vulnerabilidade_portos', w=7, h=5.5)
fig.draw()
plt.show()

---
## Figura 11 — FOB exposto a anomalias: Top 15 cadeias produtivas
Ranking das cadeias (CUCI) com maior valor de exportação durante meses anômalos.

In [ ]:
# ── Figura 11: FOB exposto por cadeia (horizontal bar) ────────────────────
fob_cad = cuci_rank_fob.sort_values('fob_exposto', ascending=False).head(15).copy()
fob_cad['fob_bi'] = fob_cad['fob_exposto'] / 1e9
fob_cad['label'] = fob_cad['NO_CUCI_GRUPO'].str[:35]
fob_cad['label'] = pd.Categorical(fob_cad['label'],
                                     categories=fob_cad.sort_values('fob_bi')['label'].tolist(),
                                     ordered=True)

# Color by HHI portuário
fig = (
    ggplot(fob_cad, aes(x='label', y='fob_bi', fill='hhi_portuario'))
    + geom_col(width=0.7, alpha=0.85)
    + coord_flip()
    + scale_fill_gradient2(
        low=PAL['green'], mid='#f0e442', high=PAL['red'],
        midpoint=0.4, name='HHI portuário',
        limits=(0.1, 0.9),
    )
    + geom_text(aes(label='fob_bi'), format_string='US$ {:.1f} bi',
                ha='left', size=7, nudge_y=0.5)
    + labs(
        x='',
        y='FOB exposto a anomalias (US$ bilhões)',
        title='Top 15 cadeias produtivas por FOB exposto a anomalias',
        subtitle='Cor indica concentração portuária (HHI) — vermelho = mais concentrado',
    )
    + theme_academic
    + theme(
        figure_size=(8, 5.5),
        legend_position='bottom',
    )
)

save_fig(fig, 'fig11_fob_exposto_cadeias', w=8, h=5.5)
fig.draw()
plt.show()

---
## Figura 12 — FOB exposto por UF de origem
Treemap mostrando o valor exportado exposto a anomalias por unidade da federação.

In [ ]:
# ── Figura 12: FOB exposto por UF (treemap via squarify) ──────────────────
try:
    import squarify
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'squarify', '-q'])
    import squarify

fob_uf = fob_exp_uf[fob_exp_uf['fob_exposto'] > 0].sort_values('fob_exposto', ascending=False).copy()
fob_uf['fob_bi'] = fob_uf['fob_exposto'] / 1e9
fob_uf['pct_exp_pct'] = fob_uf['pct_exposto'] * 100

# Color by pct_exposto
from matplotlib.colors import Normalize
norm = Normalize(vmin=15, vmax=55)
cmap_tm = LinearSegmentedColormap.from_list('tm',
    [PAL['green'], '#f0e442', PAL['red']])

fig, ax = plt.subplots(figsize=(8, 5))

values = fob_uf['fob_bi'].tolist()
labels = [f"{row['SG_UF_NCM']}\nUS$ {row['fob_bi']:.1f} bi\n({row['pct_exp_pct']:.0f}% exposto)"
          if row['fob_bi'] > 10 else row['SG_UF_NCM']
          for _, row in fob_uf.iterrows()]
colors = [cmap_tm(norm(row['pct_exp_pct'])) for _, row in fob_uf.iterrows()]

squarify.plot(sizes=values, label=labels, color=colors, alpha=0.8,
              ax=ax, text_kwargs={'fontsize': 8, 'fontfamily': 'serif'},
              pad=True, edgecolor='white', linewidth=2)

ax.set_title('FOB exposto a anomalias por UF de origem',
             fontsize=11, fontweight='bold', loc='left')
ax.axis('off')

# Colorbar
sm = plt.cm.ScalarMappable(cmap=cmap_tm, norm=norm)
cbar = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.02, aspect=15)
cbar.set_label('% FOB exposto', fontsize=8)
cbar.ax.tick_params(labelsize=7)

fig.text(0.99, -0.02, 'Área proporcional ao FOB exposto · Cor = % do FOB total da UF exposto a anomalias',
         ha='right', fontsize=7, color='#888')

save_mpl(fig, 'fig12_fob_exposto_uf_treemap')
plt.show()

---
## Figura 14 — FOB exposto por porto e classificação
Valor de exportação exposto a anomalias, separado por tipo de evento.

In [ ]:
# ── Figura 14: FOB exposto por porto e classificação ──────────────────────
fob_pc = fob_exp_cls.copy()
fob_pc['fob_bi'] = fob_pc['fob_exposto'] / 1e9

# Total per porto for ordering
porto_totals = fob_pc.groupby('porto')['fob_bi'].sum().sort_values(ascending=True)
fob_pc['porto'] = pd.Categorical(fob_pc['porto'],
                                   categories=porto_totals.index.tolist(),
                                   ordered=True)

cls_map = {'global': 'Global', 'nacional': 'Nacional', 'isolado': 'Isolado'}
fob_pc['cls_label'] = fob_pc['classificacao'].map(cls_map)
fob_pc['cls_label'] = pd.Categorical(fob_pc['cls_label'],
                                       categories=['Global', 'Nacional', 'Isolado'],
                                       ordered=True)

fig = (
    ggplot(fob_pc, aes(x='porto', y='fob_bi', fill='cls_label'))
    + geom_col(width=0.7, alpha=0.85)
    + coord_flip()
    + scale_fill_manual(
        values={'Global': PAL['red'], 'Nacional': PAL['orange'], 'Isolado': PAL['blue']},
        name='Classificação'
    )
    + labs(
        x='',
        y='FOB exposto (US$ bilhões)',
        title='FOB exposto a anomalias por porto e classificação',
        subtitle='Cruzamento temporal: meses anômalos × fluxos ComexStat (2014–2025)',
    )
    + theme_academic
    + theme(
        figure_size=(7, 5.5),
        legend_position='bottom',
    )
)

save_fig(fig, 'fig14_fob_porto_classificacao', w=7, h=5.5)
fig.draw()
plt.show()

---
## Figura 15 — Distribuição temporal das anomalias
Contagem de anomalias por semana ao longo da série, com destaque para eventos marcantes.

In [ ]:
# ── Figura 15: Timeline de anomalias (por semana) ─────────────────────────
anom_ts = anomalias.copy()
anom_ts['week'] = anom_ts['date'].dt.to_period('W').dt.start_time

# Count by week and classification
weekly = anom_ts.groupby(['week', 'classificacao']).size().reset_index(name='n')

cls_map = {'global': 'Global', 'nacional': 'Nacional', 'isolado': 'Isolado'}
weekly['cls_label'] = weekly['classificacao'].map(cls_map)
weekly['cls_label'] = pd.Categorical(weekly['cls_label'],
                                       categories=['Global', 'Nacional', 'Isolado'],
                                       ordered=True)

# Events for annotation
EVENTS_ANNOT = [
    (pd.Timestamp('2018-05-21'), 'Greve\nCaminhoneiros'),
    (pd.Timestamp('2020-03-15'), 'COVID-19'),
    (pd.Timestamp('2022-03-01'), 'Ucrânia /\nLockdowns China'),
    (pd.Timestamp('2023-12-01'), 'Ataques\nHouthi'),
    (pd.Timestamp('2024-05-15'), 'Enchentes RS'),
]

fig = (
    ggplot(weekly, aes(x='week', y='n', fill='cls_label'))
    + geom_col(width=6, alpha=0.8, position='stack')
    + scale_fill_manual(
        values={'Global': PAL['red'], 'Nacional': PAL['orange'], 'Isolado': PAL['blue']},
        name='Classificação'
    )
    + labs(
        x='Data',
        y='Anomalias por semana',
        title='Distribuição temporal das anomalias detectadas (2015–2025)',
    )
    + theme_academic
    + theme(
        figure_size=(8, 3.5),
        legend_position='bottom',
    )
)

# Save base and add annotations with matplotlib
p = fig.draw()
mpl_fig = p
ax = mpl_fig.axes[0]
ymax = ax.get_ylim()[1]

for ev_date, ev_label in EVENTS_ANNOT:
    ax.annotate(ev_label, (ev_date, ymax * 0.85),
                fontsize=6.5, ha='center', color=PAL['dark'],
                style='italic',
                arrowprops=dict(arrowstyle='->', color='#999', lw=0.5),
                xytext=(ev_date, ymax * 1.1))

save_mpl(mpl_fig, 'fig15_timeline_anomalias')
plt.show()